# Download libraries and log into Hugging face

In [ ]:
!pip install --upgrade transformers torch datasets torchvision --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.1/797.1 MB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 101.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 39.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 124.2/124.2 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 196.0/196.0 MB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.2/176.2 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.1/99.1 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
!pip install datasets seqeval evaluate huggingface_hub --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 3.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.7 MB/s eta 0:00:00


In [ ]:
from huggingface_hub import notebook_login

# You need to log in with your Hugging Face write token otherwise you can't add models to your account, when you run this cell, there is a link with 'Tokens page' here you can log in and make a token so hugging face has access to your account
notebook_login()

# Read Data and pre-process ready to train

In [ ]:
# Here we are opening the file and reading from it, I put this into a function so I can call it for the test, train and validation sets.
def open_file_get_data_bios(filepath):
    # This function returns two matrices, the words and the labels split into sentences, there is one label per word in this sentence and we will have to re-construct this when we do inference on the models because the models make predictions per token, not per word.
    words = []
    labels = []
    with open(filepath, 'r', encoding='utf-8') as file:
        # Here we make an array to add the current example, the other array is for all of the examples, so it is a 2D array where there is the number of examples and then the number of words per exmaple, which is then further divided into individual words.
        word = []
        label = []
        counter = 0

        # Here I am reading the file line by line, a line is 'Admission 100130 0 9 O' For example, here we only care about two things, the word, which and the label, we can extract these by calling line.split which gets rid of spaces and puts everything into a matrix so to get the first and last element, we can simply say [0] and [-1] to get the last element.
        for line in file:
            split_lines = line.split()

            # You need to make sure that when reading the data that there are no white gaps, if you see the data after every full stop there are gaps so we can just ignore them.
            if len(split_lines) > 0:

                # Here, every time a word is read, we add one to a counter and move onto the next word, there is a max of 128 words and this is because they are un-tokenized. So when you do tokenize them pretty much any value bigger than 128 would mean that there are more than 512 tokens and that is maximum number of tokens for most BERT models and you would just truncate the input, loosing data in the process. I also check that after 100 words if there are any full stops so that there is a nice end to the data but the data is not too short. You could also end the input after 3 sentences say, so after 3 full stops, but this would mean that the input is very variable which is not ideal for training but something that could be tested.
                if counter == 128 or (split_lines[0] == "." and counter > 100):

                    # Here I make sure that I add the full stop when the example ends at a full stop, this could have been re-formated to just be after the word.append() at the bottom
                    if len(split_lines) != 0:
                        if split_lines[0] == ".":
                            word.append(split_lines[0])
                            label.append(split_lines[-1])
                    if len(word) != 0:
                        words.append(word)
                        labels.append(label)
                    word = []
                    label = []
                    counter = 0
                    continue

                word.append(split_lines[0])
                label.append(split_lines[-1])
                counter += 1
    return words, labels

# Here I call the function three times on the three different datasets. Make sure to have the three datasets in the files section of your google co-lab.
words, labels = open_file_get_data_bios("train_spacy.txt")
wordsTest, labelsTest = open_file_get_data_bios("test_spacy.txt")
wordsValid, labelsValid = open_file_get_data_bios("valid_spacy.txt")

# Here I add the train to the validaton to have more data to train on, only do this after you have your hyperparameters for your models. You should use the validation set as your test set and then train your validation + train afterwards.
words = words + wordsValid
labels = labels + labelsValid

In [ ]:
# Showing the structure of the data
print(words[0])

['Admission', 'Date', ':', '[', '*', '*', '2176', '-', '7', '-', '5', '*', '*', ']', 'Discharge', 'Date', ':', '[', '*', '*', '2176', '-', '7', '-', '7', '*', '*', ']', 'Service', ':', 'SURGERY', 'Allergies', ':', 'Amoxicillin', '/', 'Penicillins', '/', 'Coumadin', '/', 'Oxycodone', '/', 'Megestrol', 'Acetate', '/', 'Remeron', '/', 'Ritalin', 'Attending:[**First', 'Name3', '(', 'LF', ')', '301', '*', '*', ']', 'Chief', 'Complaint', ':', 'Free', 'air', 'on', 'CXR', 'Major', 'Surgical', 'or', 'Invasive', 'Procedure', ':', 'None', 'History', 'of', 'Present', 'Illness', ':', 'Pt', 'is', 'a', '87', 'y', '/', 'o', 'male', 'with', 'extensive', 'past', 'medical', 'history', 'who', 'was', 'recently', 'discharged', 'after', 'admission', 'for', 'possible', 'meningitis', '/', 'altered', 'mental', 'status', '.']


In [ ]:
# Here is a dictionary showing the mapings from the labels to the numbers, as models only read tensors we will need to convert the labels to numbers
label_to_tag = {
    'O':0,
    'B-Drug':1, 'I-Drug':2,
    'B-Reason':3, 'I-Reason':4,
    'B-Route':5, 'I-Route':6,
    'B-Strength':7, 'I-Strength':8,
    'B-Form':9, 'I-Form':10,
    'B-Dosage':11, 'I-Dosage':12,
    'B-Frequency':13, 'I-Frequency':14,
    'B-Duration':15, 'I-Duration':16,
    'B-ADE':17, 'I-ADE':18,
}

# We also need just the names so when we do the metrics we can convert the labels back into numbers
label_names = [name for name in label_to_tag]

# This is a dictionary of the inverse of label_to_tag, so {"0": "O", "1": "B-Drug", etc} we need to give this to the model as it is training so that it can be uploaded into Hugging Face, we don't actually use it in this script but Hugging Face needs it when uploading models.
id_to_label = {i:v for i, v in enumerate(label_to_tag)}

In [ ]:
# Here I run a double for loop to cycle through every label and turn it from string to number, 0-18
labels_num = [[label_to_tag[lab] for lab in label] for label in labels]
labels_numTest = [[label_to_tag[lab] for lab in label] for label in labelsTest]
labels_numValid = [[label_to_tag[lab] for lab in label] for label in labelsValid]

In [ ]:
# This step is very important, when you tokenize the incoming string, it adds more items to the words, therefore we need to have that reflected in the labels, if you imagine the word 'paracetamol' it will be tokenized like 'para' '##ce' '##ta' '##mol' or something similar to that. The label 'B-Drug' needs to be expanded to be 'B-Drug' and then every sub-token as 'I-Drug'. And if it is 'O' then add more 'O's. You can just add one to the label if the label is odd, this means if you look at the dictionary that it is 'B-' so you can add 'I-' afterwards by simply adding 1 to the label.
# To really understand this you need to follow the https://huggingface.co/learn/nlp-course/chapter7/2?fw=pt it is very important to follow this in your own notebook from scratch and then come back to this.
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    current_word = None
    for word_id in word_ids:
        if word_id is None:
            new_labels.append(-100)
        elif word_id != current_word:
            new_labels.append(labels[word_id])
        else:
            label = labels[word_id]
            new_labels.append(label if label % 2 == 0 else label + 1)
        current_word = word_id
    return new_labels

# Train Models

In [ ]:
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
import numpy as np
import evaluate

metric = evaluate.load("seqeval")

# This was from the huggingface tutorial (https://huggingface.co/learn/nlp-course/chapter7/2?fw=pt) and it is needed for Hugging Face to evaluate the model on the test dataset.
def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)

    true_labels = [[label_names[l] for l in label if l != -100] for label in labels]
    true_predictions = [[label_names[p] for (p, l) in zip(prediction, label) if l != -100] for prediction, label in zip(predictions, labels)]
    all_metrics = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": all_metrics["overall_precision"],
        "recall": all_metrics["overall_recall"],
        "f1": all_metrics["overall_f1"],
        "accuracy": all_metrics["overall_accuracy"],
    }

In [ ]:
from transformers import AutoTokenizer, DataCollatorForTokenClassification, AutoModelForTokenClassification, TrainingArguments, Trainer
from datasets import Dataset
import torch.nn as nn

# This is the main part of the train loop but we have done most of the steps to get there now we need to download the base model and feed the data that we have pre-processed through it now.
# Here because of the ensemble of models I am training 8 models in one go, it is unlikely that you need that many so you can change this to be just 1 if you want by reducing the size of the matrix to just one, also change the name.
# model_checkpoint = ["google-bert/bert-base-uncased",
#                     "dmis-lab/biobert-base-cased-v1.2",
#                     "medicalai/ClinicalBERT",
#                     "emilyalsentzer/Bio_ClinicalBERT",
#                     "microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext",
#                     "allenai/biomed_roberta_base",
#                     "FacebookAI/roberta-base",
#                     "FacebookAI/roberta-large"]

# model_names = ["BERT",
#                "BioBERT",
#                "ClinicalBERT",
#                "BioClinicalBERT",
#                "PubMedBERT",
#                "BioMedRoBERTa",
#                "RoBERTa",
#                "RoBERTa-Large"]

model_checkpoint = ["FacebookAI/roberta-large"]

model_names = ["RoBERTa-Large"]

# Cycle through the models and train them, then upload them to Hugging Face
for i, model in enumerate(model_checkpoint):
    # Grab the tokenizer, be careful as different BERT models have different tokenizers, so always use the same Hugging Face directory for both the tokenizer and the model itself.
    tokenizer = AutoTokenizer.from_pretrained(model, add_prefix_space=True)

    # Download the model itself and pass it in two dictionaries, the id2label and the label2id, which we have made earlier.
    model = AutoModelForTokenClassification.from_pretrained(
        model,
        id2label=id_to_label,
        label2id=label_to_tag,
    )

    # Here we tokenize the dataset with a function, we pass this function in the dataset.map as it is what Hugging Face expects the dataset to be like.
    def tokenize_and_align_labels(examples):
        # Here we tokenize the input as we make sure to add is_split_into_words=True as the words are in the format where each word is it's own item in the matrix.
        tokenized_inputs = tokenizer(examples["tokens"], truncation=True, is_split_into_words=True)

        # Here we make sure that the labels are the same size as the tokens, make sure to follow the tutorial to understand this
        tokenized_inputs["labels"] = [align_labels_with_tokens(ner_tags, tokenized_inputs.word_ids(i)) for i, ner_tags in enumerate(examples["ner_tags"])]
        return tokenized_inputs

    # We turn our dataset into a standard format that hugging face expects
    dataset = Dataset.from_dict({"tokens": words, "ner_tags": labels_num})
    datasetTest = Dataset.from_dict({"tokens": wordsTest, "ner_tags": labels_numTest})
    datasetValid = Dataset.from_dict({"tokens": wordsValid, "ner_tags": labels_numValid})

    # We can then use the .map function inside of the dataset to tokenize the dataset and also to remove the un-tokenized data
    tokenized_dataset = dataset.map(tokenize_and_align_labels, batched=True, remove_columns=dataset.column_names)
    tokenized_datasetTest = datasetTest.map(tokenize_and_align_labels, batched=True, remove_columns=datasetTest.column_names)
    tokenized_datasetValid = datasetValid.map(tokenize_and_align_labels, batched=True, remove_columns=datasetValid.column_names)

    # For batching
    data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

    # I did tests where I trained just the classifier first and then the rest of the model, in the end it took too much time but it defenetly improved the F-1 score. When you have a random classifier at the start of training, the derivatives into the model also also optimizing for noise as the classifier is instantiated as random values, so having the model make meaningful predictions before training the base model makes the finetuning better.
    # for param in model.base_model.parameters():
    #     param.requires_grad = False

    # I made the drop-out slightly bigger so that the predictions are more robust
    model.dropout = nn.Dropout(0.2)

    for param in model.parameters():
        param.data = param.data.contiguous()

    # Experiment with these and see what changes, they are pretty good settings as they are but do see how changing them affects the model.
    args = TrainingArguments(
        f"{model_names[i]}-full-finetuned-ner-pablo",
        evaluation_strategy="epoch",
        # eval_steps=1000,
        save_strategy="epoch",
        # save_steps=1000,
        learning_rate=0.00005, #0.0002, 0.0001, 0.00005
        num_train_epochs=4,
        weight_decay=0.01,
        per_device_train_batch_size=32, #16, #32
        per_device_eval_batch_size=32,
        dataloader_num_workers=4,
        dataloader_pin_memory=True,
        optim="adamw_torch",
        fp16=True,
        warmup_ratio=0.1,
        push_to_hub=True,
        # load_best_model_at_end=True,
        # metric_for_best_model="f1",
        # greater_is_better=True,
        #lr_scheduler_type='constant'
    )

    # Here we actually make the trainer with the model, the hyperparameters, the data the eval data, etc.
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tokenized_dataset,
        eval_dataset=tokenized_datasetTest,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        tokenizer=tokenizer,
    )

    # Call it to train, this is what takes the longest of course.
    trainer.train()

    # Once it is done, upload it to Hugging face, after this is done it moves onto the next model to train.
    trainer.push_to_hub(commit_message="Training complete")

/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

...

Map:   0%|          | 0/4843 [00:00<?, ? examples/s]

Map:   0%|          | 0/714 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1525: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/accelerate/accelerator.py:488: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self.scaler = torch.cuda.amp.GradScaler(**kwargs)
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.080950,0.794722,0.763787,0.778948,0.976373
2,No log,0.071258,0.809653,0.792889,0.801183,0.977947
3,0.221300,0.070437,0.809238,0.804569,0.806897,0.977981
4,0.221300,0.073850,0.811342,0.808168,0.809752,0.978190


/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
/usr/lib/python3.10/multiprocessing/popen_fork.py:66: RuntimeWarning: os